Fonte = https://dados.ons.org.br/dataset/geracao-usina-2


Perguntas de Pesquisa: 
 - É possível prever a geração de energia hidrelétrica utilizando dados históricos de geração?
 - Quais períodos do ano apresentam maior e menor geração de energia hidrelétrica?
 - O nível dos reservatórios está relacionado ao aumento do valor cobrado pela energia?

Hipoteses: 
 - O histórico de geração permite prever a geração futura
 -A geração possui padrões sazonais
 -O nível dos reservatórios está relacionado à geração
 -Em epocas de estiagem, o valor da energia aumenta por causa da escassez



In [ ]:
import pandas as pd

df = pd.read_csv('data.csv', sep=';')
df.head()

In [ ]:
df.describe()

Variavel-alvo: val_geracao
Tipo de geração: Hidráulica
Regiao: Sul e Centro-Oeste
Granularidade: Horária
Período 2001 - 2020
Horizonte de previsão: Mensal

Por que escolher a Granularidade Mensal: 
    O conjunto de dados do ONS ja possui os registros, o que permite aproveitar a resolucaos. A geracao hidreletrica pode variar ao longo do tempo, e a granularidade mensal permitiria analisar e identificar isso de uma forma mais detalhada. Isso nos permitiria, alem de tudo, avaliar o impacto financeiro ao longo do tempo.

Horizonte de planejamento para o mes seguinte:
    Permite estimar a geracao do mes seguinte
    Possibilita utilizar os padroes e horarios para estimar e identificar tendencias. 
    Parece um horizonte adequado para um modelo inicial, sem tornar o problema muito complexo.

Nao foi possivel fazer a agregacao dos dados com o pandas porque devido ao grande volume de dados, o processamento apresentou limitacoes de meemoria. Diante disso, optou-se pelo uso do DuckDB.
Foi realizado o processo de juncao dos 19 anos em um unico dataframe.

In [ ]:
import os
import duckdb

os.makedirs("dados_consolidados", exist_ok=True)
con = duckdb.connect()

In [ ]:
for ano_inicio in range(2001, 2021, 2):

    ano_fim = ano_inicio + 1

    arquivos = [
        f"dados_mensais/GERACAO_USINA-2_{ano_inicio}.parquet",
        f"dados_mensais/GERACAO_USINA-2_{ano_fim}.parquet"
    ]

    destino = f"dados_consolidados/geracao_{ano_inicio}_{ano_fim}.parquet"
    con.execute(f"""
        COPY (
        SELECT *
        FROM read_parquet({arquivos})
        )
        TO '{destino}'
        (FORMAT PARQUET, COMPRESSION ZSTD)
        """)

In [ ]:
con.execute("""
    COPY (
    SELECT *
    FROM read_parquet('dados_consolidados/*.parquet')
    )
    TO 'geracao2001_2020.parquet'
    (FORMAT PARQUET, COMPRESSION ZSTD)
    """)

In [ ]:
import duckdb
con = duckdb.connect()

df = con.execute("""
SELECT *
FROM read_parquet('geracao2001_2020.parquet')
ORDER BY din_instante DESC
LIMIT 10
""").df()

df

Como vamos fazer uma analise Mensal, a seguir, se cria uma coluna com o ano e uma com o mes. 

In [ ]:
con.execute("""
    COPY (
    SELECT *,
    EXTRACT(YEAR FROM din_instante) AS Ano
    FROM read_parquet('geracao2001_2020.parquet')
    )
    TO 'geracao_modificada.parquet'
    (FORMAT PARQUET, COMPRESSION ZSTD)
""")

In [ ]:
con.execute("""
    COPY(
    SELECT *,
    EXTRACT (MONTH FROM din_instante) AS MES
    FROM read_parquet('geracao_modificada.parquet')
    )
    TO 'geracao_temp.parquet'
    (FORMAT PARQUET, COMPRESSION ZSTD)
""")

In [ ]:
import os
os.replace(
    'geracao_temp.parquet',
    'geracao_modificada.parquet'
)

Agora o arquivo modificado contem o ano e o mes.